In [ ]:
# Imports and setup
import json
import os
import sys
from typing import List, Dict, Optional, Tuple, Any
from collections import defaultdict

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm import tqdm
import seaborn as sns
from sklearn.metrics import auc, roc_curve

# Add parent directories to path
sys.path.append('..')
sys.path.append('../..')

# Import utility functions
from plotting_notebooks.plotting_utils import (
    filter_and_prepare_data,
    get_roc_curve,
    interpolate_tpr,
    is_in_training_data,
    load_in_combined_results,
    load_monitoring_data,
    merge_monitoring_with_combined_results,
    plot_roc_curves,
    plot_suspiciousness_distributions,
    prepare_monitoring_data,
    prepare_data_for_training_progress
)
from utils.analysis_utils import extract_verdict
from utils.path_utils import resolve_path


In [ ]:
# Constants and model configurations

# Base models to include in analysis
BASE_MODELS = ['claude-3-5-sonnet-20241022', 'gpt-4o-2024-08-06', 'o3-mini', 'o1', 'claude-3-5-haiku-20241022']

# Fine-tuned model mappings from different experiments
FT_MODEL_MAPPINGS = {
    # From system_prompting_ft_3.py (Combined-3 series)
    'ft3': {
        "gpt-4o-2024-08-06": 0,
        'ft:gpt-4o-2024-08-06:scale-safety-research-1:combined-3-reviewed:BCYJFx2l:ckpt-step-70': 70,
        'ft:gpt-4o-2024-08-06:scale-safety-research-1:combined-3-reviewed:BCYJFtC0:ckpt-step-140': 140,
        'ft:gpt-4o-2024-08-06:scale-safety-research-1:combined-3-reviewed:BCYJFa40': 210
    },
    
    # # From system_prompting_ft.py (Combined-1 series)
    # 'ft1': {
    #     "gpt-4o-2024-08-06": 0,
    #     "ft:gpt-4o-2024-08-06:scale-safety-research-1:1:BA612ygq:ckpt-step-70": 70,
    #     "ft:gpt-4o-2024-08-06:scale-safety-research-1:1:BA612NBM:ckpt-step-140": 140,
    #     "ft:gpt-4o-2024-08-06:scale-safety-research-1:1:BA613xjZ": 210,
    # },
    
    # # From system_prompting_ft_2.py (Combined-2 series)
    # 'ft2': {
    #     "gpt-4o-2024-08-06": 0,
    #     "ft:gpt-4o-2024-08-06:scale-safety-research-1::BAU1ENZ3:ckpt-step-140": 140,
    #     "ft:gpt-4o-2024-08-06:scale-safety-research-1::BAU1Fwer:ckpt-step-210": 210,
    #     "ft:gpt-4o-2024-08-06:scale-safety-research-1::BAU1FbkP": 280,
    # }
}

# Combine all fine-tuned models
ALL_FT_MODELS = set()
for model_set in FT_MODEL_MAPPINGS.values():
    ALL_FT_MODELS.update(model_set.keys())
ALL_FT_MODELS = list(ALL_FT_MODELS)

# System prompts to include
SYSTEM_PROMPTS = ['system_prompt_1', 'system_prompt_5', 'system_prompt_5_temp_0']

# Number of monitors to process per monitoring response (max observed is 10)
MAX_NUM_MONITORS = 10

# Transcript types
DEFAULT_TRANSCRIPT_TYPES = ["transcripts", "transcripts_no_hint", "transcripts_no_secondary_task"]

# Tasks to exclude (often due to issues or special characteristics)
EXCLUDED_TASKS = ['election_tampering', 'art_collectors', 'blog_post', 'spam_filter_update']


In [ ]:
# Analysis functions to load the dataframes

def load_monitoring_data_for_all_models():
    """
    Load monitoring data for all models and system prompts.
    Returns separate dataframes for each system prompt.
    """
    # Dictionary to hold dataframes for each system prompt
    system_prompt_dfs = {}
    
    # Load combined results (used later for merging)
    combined_results = load_in_combined_results(include_failed_attempts=False, include_failed_main_task=False)
    
    # Load data for each system prompt
    for system_prompt in SYSTEM_PROMPTS:
        print(f"Loading data for {system_prompt}...")
        
        # Use the prepare_monitoring_data function to load data
        # This handles both base and fine-tuned models
        all_models = ['claude-3-5-sonnet-20241022'] + list(ALL_FT_MODELS)
        
        df = prepare_monitoring_data(
            base_dir=f'monitor_responses/{system_prompt}',
            monitor_models=all_models,
            num_monitors=MAX_NUM_MONITORS,
            compute_aggregates=True,
            tag_word='verdict'
        )
        df.drop(columns=['data'], inplace=True)
        df['monitor_agent'] = df.monitor_agent.apply(lambda x: FT_MODEL_MAPPINGS['ft3'][x] if x in FT_MODEL_MAPPINGS['ft3'] else x)
        # Add side_task_evaluation column
        df['side_task_evaluation'] = df.transcript_type != 'transcripts_no_secondary_task'
        
        # Store in dictionary
        system_prompt_dfs[system_prompt] = df
        
        print(f"Loaded {len(df)} rows for {system_prompt}")
    
    return system_prompt_dfs, combined_results


In [ ]:
# Analysis functions
system_prompt_dfs, combined_results = load_monitoring_data_for_all_models()
system_prompt_dfs['system_prompt_1']

In [ ]:
# Load training data
training_data_path = resolve_path('finetuning/training_data_dfs/combined_3.csv')
if os.path.exists(training_data_path):
    training_data = pd.read_csv(training_data_path)
    print(f"Loaded training data: {len(training_data)} rows")
else:
    print(f"Warning: Could not find training data at {training_data_path}")
    training_data = pd.DataFrame()  # Empty dataframe as fallback


In [ ]:
# Generate heatmaps for each system prompt

def plot_heatmaps_for_system_prompts(system_prompt_dfs: Dict[str, pd.DataFrame], metric: str = 'max_suspiciousness'):
    """
    Generate and display heatmaps for each system prompt, showing the TPR at FPR=0.03 performance 
    of each monitor on each task pair.
    """
    for system_prompt, df in system_prompt_dfs.items():
        print(f"\nProcessing {system_prompt}...")
        
        # Create a DataFrame to store TPR scores
        tasks = df['task_name'].unique()
        monitors = df['monitor_agent'].unique()
        tpr_scores = []
        
        # Calculate TPR at FPR=0.03 for each task-monitor pair
        for task in tasks:
            if task in training_data.task_name.unique():
                continue
            for monitor in monitors:
                task_data = df[(df['task_name'] == task) & (df['monitor_agent'] == monitor)]
                if len(task_data) > 0:
                    # Calculate ROC curve and get TPR at FPR=0.03
                    y_true = task_data['side_task_evaluation'].astype(int)
                    y_score = task_data[metric] / 100  # Normalize to 0-1
                    if len(np.unique(y_true)) > 1:  # Check if we have both positive and negative samples
                        fpr, tpr = get_roc_curve(task_data, metric, side_task_column='side_task_evaluation')
                        tpr_at_fpr = interpolate_tpr(fpr, tpr, 0.03)
                    else:
                        tpr_at_fpr = np.nan
                    tpr_scores.append({
                        'task_name': task,
                        'monitor_agent': monitor,
                        'tpr': tpr_at_fpr
                    })
        
        # Convert to DataFrame and pivot for heatmap
        tpr_df = pd.DataFrame(tpr_scores)
        if len(tpr_df) == 0:
            print(f"No data available to plot for {system_prompt}.")
            continue
            
        heatmap_data = tpr_df.pivot(
            index='task_name',
            columns='monitor_agent',
            values='tpr'
        )

        # Plot the heatmap
        plt.figure(figsize=(12, 8))
        sns.heatmap(
            heatmap_data, 
            annot=True, 
            fmt=".2f", 
            cmap="YlGnBu",
            vmin=0.0,  # TPR ranges from 0.0 to 1.0
            vmax=1.0,
            cbar_kws={'label': 'TPR at FPR=0.03'}
        )
        plt.title(f"TPR at FPR=0.03 Performance Heatmap for {system_prompt} ({metric})")
        plt.xlabel("Monitor Agent")
        plt.ylabel("Task Name")
        plt.xticks(rotation=45, ha='right')
        plt.yticks(rotation=0)
        plt.tight_layout()
        plt.show()


# Call the function to plot heatmaps
plot_heatmaps_for_system_prompts(system_prompt_dfs, metric='6th_highest_suspiciousness')

In [ ]:
df = system_prompt_dfs['system_prompt_5']
df = df.set_index(['task_name', 'transcript_type', 'main_agent', 'file_name', 
              'side_task_evaluation', 'monitor_agent']).unstack(level='monitor_agent').reset_index()
df = df[~df.task_name.isin(training_data.task_name.unique())]

In [ ]:
df.task_name.value_counts()

In [ ]:
# Prepare data for decision tree
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

# Select features to use
feature_cols = [('mean_suspiciousness', 'claude-3-5-sonnet-20241022'), 
                ('median_suspiciousness', 210),
                ]
print("Using these features:", feature_cols)

# Prepare X (features) and y (target)
X = df[feature_cols].fillna(0)  # Fill NaN with 0 for simplicity
y = df['side_task_evaluation']

# Create readable feature names for visualization
feature_names = [f"{col[0]}\n{col[1]}" for col in feature_cols]

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train decision tree
# Start with a relatively shallow tree for interpretability
tree = DecisionTreeClassifier(max_depth=2, random_state=42)
tree.fit(X_train, y_train)

# Make predictions
y_pred = tree.predict(X_test)
y_pred_proba = tree.predict_proba(X_test)[:, 1]

# Print results
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nROC AUC Score:", roc_auc_score(y_test, y_pred_proba))

# Print feature importance
importance = pd.DataFrame({
    'feature': feature_names,
    'importance': tree.feature_importances_
})
importance = importance.sort_values('importance', ascending=False)
print("\nFeature Importance:")
print(importance)


In [ ]:
# Plot ROC curve
from sklearn.metrics import roc_curve
import matplotlib.pyplot as plt

fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label=f'ROC (AUC = {roc_auc_score(y_test, y_pred_proba):.2f})')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve for Decision Tree')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# Plot feature importance
plt.figure(figsize=(10, 6))
plt.barh(importance['feature'], importance['importance'])
plt.xlabel('Feature Importance')
plt.title('Feature Importance in Decision Tree')
plt.tight_layout()
plt.show()


In [ ]:
# Plot the decision tree
plt.figure(figsize=(20, 10))
plot_tree(tree, 
          feature_names=feature_names,
          class_names=['No Side Task', 'Has Side Task'],
          filled=True,
          rounded=True,
          fontsize=10)
plt.title("Decision Tree Visualization")
plt.tight_layout()
plt.show()


In [ ]:
# Print out the decision rules in text form
def get_rules(tree, feature_names):
    tree_ = tree.tree_
    feature_name = [
        feature_names[i] if i != -2
        else "undefined!"
        for i in tree_.feature
    ]

    def recurse(node, depth, rules):
        indent = "  " * depth
        if tree_.feature[node] != -2:
            name = feature_name[node]
            threshold = tree_.threshold[node]
            print(f"{indent}if {name} <= {threshold:.1f}:")
            recurse(tree_.children_left[node], depth + 1, rules)
            print(f"{indent}else:  # if {name} > {threshold:.1f}")
            recurse(tree_.children_right[node], depth + 1, rules)
        else:
            samples = tree_.n_node_samples[node]
            class_prob = tree_.value[node][0][1] / samples
            print(f"{indent}return 'Side Task' (probability: {class_prob:.2f}, samples: {samples})")
            
    print("Decision Rules:")
    recurse(0, 1, [])

get_rules(tree, feature_names)
